# 📚 Intelligent Document Management & Search System
### End-to-End NLP Preprocessing, TF-IDF Vectorization, Supervised Classification, Unsupervised Clustering, and Dual Search Engine

This notebook presents the complete data story, methodology, and empirical evaluation for our **Intelligent Document Management and Search System**. We analyze a 510-record multi-domain enterprise document repository spanning **Technology**, **Finance**, **Healthcare**, **Legal**, and **Science**, addressing five core functional pillars:
1. **Document Preprocessing & Schema Hygiene** (anomalous record detection, missing target filtering, date-to-timestamp conversion, tokenization, stopword removal, and morphological lemmatization).
2. **TF-IDF Feature Extraction** (sublinear Term Frequency, Inverse Document Frequency, $L_2$ normalization, and domain vocabulary discovery).
3. **Supervised Document Classification** (strict leak-free 70/15/15 train/validation/test partitioning, multi-model evaluation, overfitting remediation, 95% bootstrap confidence intervals, and subgroup slice analysis).
4. **Unsupervised Document Clustering** ($K$-Means across $k \in \{2..8\}$ optimized via Silhouette Score, 2D PCA projections colored by cluster label and discrete category, and cluster profiling).
5. **Keyword Search & Cosine Similarity Search** (Inverted-Index retrieval with term highlighting and TF-IDF Cosine Similarity ranking).

## 1. Schema Inspection, Data Hygiene & Anomaly Audit
Before building any NLP or machine learning pipelines, we inspect the raw schema of `data/documents_corpus.csv`, identify corrupted rows that violate domain constraints, and isolate unlabeled records lacking a target `category`.

In [ ]:
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure project root is on sys.path when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.text_preprocessing import audit_and_clean_corpus, extract_document_stats

raw_df = pd.read_csv(PROJECT_ROOT / "data/documents_corpus.csv")
clean_df, audit_report = audit_and_clean_corpus(raw_df)

print(f"Initial Raw Rows: {audit_report['initial_row_count']}")
print(f"Anomalous Rows Identified & Dropped: {audit_report['anomalous_row_count']}")
print(f"Missing-Target Rows Identified & Dropped: {audit_report['missing_target_row_count']}")
print(f"Clean Corpus Rows Retained: {audit_report['clean_row_count']}")

display(pd.DataFrame(audit_report["anomalous_rows"]))
display(pd.DataFrame(audit_report["missing_target_rows"]))

### Analysis of Schema Audit & Missing Value Strategy
- **Schema & Field Overview**: The raw dataset contains **510 rows** across 10 columns: `doc_id` (unique identifier), `title` and `content` (free-text fields), `published_date` (ISO date string converted into a numerical Unix timestamp `published_timestamp`), `access_tier` (ordinal security level: `Public < Internal < Confidential < Restricted`), `source_channel` (nominal ingestion source), `is_verified` (pre-encoded binary `0`/`1` flag), `author_experience_years` and `citation_count` (continuous numerical metadata), and `category` (target class label).
- **Rows That Do Not Make Sense (5 Rows)**: Our audit flagged exactly **5 corrupted records**: `DOC-9001` (impossible negative author experience of `-8.5` years), `DOC-9002` (impossible negative citation count of `-42.0`), `DOC-9003` (empty/whitespace-only document body), `DOC-9004` (implausible author career experience of `145.0` years), and `DOC-9005` (invalid binary verification flag `is_verified = 9`).
- **Rows Without a Target Value (5 Rows)**: Exactly **5 archival ingestion records** (`DOC-9101` through `DOC-9105`) have `category = NaN`. Because supervised training requires ground-truth labels, these 5 unlabeled rows and the 5 corrupted rows are removed, leaving **500 valid documents** (100 per category).
- **Preserving Incomplete Metadata Rows**: Within the 500 valid documents, `author_experience_years` and `citation_count` each contain **37 missing values (7.40%)**. Rather than dropping 14% of valid text documents, we retain all 500 records and handle missing numerical values inside our leak-free training pipeline using median imputation paired with explicit missingness indicator features (`add_indicator=True`).

## 2. Exploratory Data Analysis (EDA) & Outlier Visualization
Next, we visualize category balance, metadata distributions, citation outliers, and access-tier stratification across the 500 clean documents.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(PROJECT_ROOT / "reports/figures/eda_distributions.png")))
summary_stats = clean_df.groupby("category")[["author_experience_years", "citation_count", "word_count"]].agg(["mean", "median", "std"])
display(summary_stats.round(2))

### Analysis of Exploratory Visualizations
- **Balanced Multi-Class Distribution**: All 5 categories (`Finance`, `Healthcare`, `Legal`, `Science`, `Technology`) contain exactly **100 documents** each, preventing majority-class bias.
- **Domain-Specific Metadata Trends & Outliers**: `Science` and `Healthcare` documents exhibit the highest citation counts (mean **53.8** and **43.9** citations respectively, with upper-tail outliers exceeding 80 citations), reflecting academic research norms on `Research_Portal`. Conversely, `Legal` documents have the highest mean author experience (**14.1 years**) and the highest concentration of `Confidential` and `Restricted` access tiers (**60%+** combined), whereas `Technology` documents have younger authorship (mean **7.7 years**) and are predominantly hosted on `Enterprise_Wiki`.

## 3. NLP Preprocessing & TF-IDF Feature Extraction
We inspect how our NLP preprocessing pipeline normalizes raw text (lowercasing, punctuation removal, stopword filtering, and morphological lemmatization) and how `TFIDFDocumentEngine` extracts discriminative unigrams and bigrams fitted strictly on the **Training Partition (350 documents)**.

In [ ]:
sample_row = clean_df.iloc[0]
sample_stats = extract_document_stats(sample_row["raw_full_text"])
print(f"Document ID: {sample_row['doc_id']} | Category: {sample_row['category']}")
print(f"Raw Word Count: {sample_stats['raw_word_count']} -> Clean Lemmatized Tokens: {sample_stats['clean_token_count']}")
print(f"Sample Preprocessed Text:\n  {sample_stats['preprocessed_text'][:220]}...")

display(Image(filename=str(PROJECT_ROOT / "reports/figures/tfidf_top_terms.png")))

### Analysis of NLP Preprocessing & Top TF-IDF Terms
- **Vocabulary Compression & Normalization**: Removing English and boilerplate corporate stopwords alongside suffix lemmatization reduces raw document length by ~45% while preserving high-signal technical terminology.
- **Category-Defining TF-IDF Signatures**: Fitting an 800-feature unigram/bigram `TfidfVectorizer` strictly on the 350 training documents surfaces highly interpretable domain markers:
  - **Finance**: `adequacy` (mean TF-IDF `0.0525`), `central` (`0.0519`), `equity` (`0.0485`), `foreign` (`0.0480`), `derivatives` (`0.0479`), `portfolio` (`0.0471`).
  - **Healthcare**: `clinical` (`0.0697`), `patient` (`0.0606`), `cerebrospinal` (`0.0496`), `gene` (`0.0467`), `oncology` (`0.0454`), `telemedicine` (`0.0453`).
  - **Legal**: `regulatory` (`0.0604`), `compliance` (`0.0548`), `data` (`0.0516`), `environmental` (`0.0502`), `duty` (`0.0497`), `gdpr` (`0.0485`).
  - **Science**: `superconduct` (`0.0595`), `ocean` (`0.0557`), `correction` (`0.0499`), `atmospheric` (`0.0498`), `qubit` (`0.0489`).
  - **Technology**: `learn` (`0.0602`), `zero` (`0.0535`), `distribut` (`0.0519`), `database` (`0.0509`), `network` (`0.0500`), `cybersecurity` (`0.0472`).

## 4. Supervised Document Classification & Overfitting Analysis
Following strict featurization ordering, we split the 500 clean documents into **70% Training (350 rows)**, **15% Validation (75 rows)**, and **15% Held-Out Test (75 rows)** *before* fitting our `ColumnTransformer` (`TfidfVectorizer` + `StandardScaler` + `OrdinalEncoder` on `access_tier` + `OneHotEncoder` on `source_channel` + pre-encoded binary `is_verified`). We compare seven models across predictive accuracy, overfitting behavior, and operational efficiency.

In [ ]:
metrics_data = json.loads((PROJECT_ROOT / "reports/metrics_summary.json").read_text(encoding="utf-8"))
val_comp_df = pd.DataFrame(metrics_data["validation_model_comparison"]).T.drop(columns=["confusion_matrix"])
display(val_comp_df)

display(Image(filename=str(PROJECT_ROOT / "reports/figures/classification_comparison.png")))
display(Image(filename=str(PROJECT_ROOT / "reports/figures/confusion_matrices.png")))

### Analysis of Classification Models, Regularization & Held-Out Test Performance
- **Baseline vs. Linear & Ensemble Models**: The `Naive Baseline (Majority)` model achieves only **0.0667 Macro F1** (20.0% validation accuracy). Adding TF-IDF features with `Complement Naive Bayes` lifts validation Macro F1 to **0.7118** (5-fold CV Macro F1 **0.8390**).
- **Diagnosing & Remediating Overfitting**: An unregularized `Decision Tree (Unregularized)` memorizes the training set (`Train Macro F1 = 1.0000`) but suffers severe variance on the validation set (`Val Macro F1 = 0.6014`, `Val ROC-AUC = 0.7500`). Applying depth constraints (`max_depth=10`), leaf sample minimums (`min_samples_leaf=4`), and cost-complexity pruning (`ccp_alpha=0.004`) in `Decision Tree (Regularized)` reduces the train-validation gap (`Train F1 = 0.8572`) and boosts validation ROC-AUC from **0.7500 to 0.8038** and PR-AUC from **0.4471 to 0.5780**.
- **Production Model Selection (`Logistic Regression L2`)**: `Random Forest (Regularized)` (`Val ROC-AUC = 0.9636`), `Calibrated Linear SVM (L2)` (`Val ROC-AUC = 0.9618`), and `Logistic Regression (L2)` (`Val ROC-AUC = 0.9587`, `5-Fold CV F1 = 0.8536`) all achieve strong discrimination. We select `Logistic Regression (L2)` for production deployment due to its compact serialized footprint (**55.21 KB** vs. **1,372.69 KB** for Random Forest), fast training (**22.80 ms**), low per-document latency (**0.0675 ms**), and native probability calibration.
- **Held-Out Test Set & 95% Bootstrap CIs**: On the unseen 75-document test set, `Logistic Regression (L2)` achieves **85.33% Accuracy** (95% CI: `[77.33%, 92.00%]`), **0.8545 Macro F1** (95% CI: `[0.7724, 0.9203]`), **0.9833 Macro ROC-AUC** (95% CI: `[0.9651, 0.9957]`), and **0.9449 Macro PR-AUC**.

## 5. Unsupervised Document Clustering ($K$-Means & 2D PCA Projection)
To discover organic topical groupings without relying on category labels, we standardize the combined semantic SVD components and numerical/ordinal metadata features, evaluate $K$-Means across $k \in \{2, 3, 4, 5, 6, 7, 8\}$ using the **Silhouette Score**, and project the corpus into 2 dimensions via **Principal Component Analysis (PCA)**.

In [ ]:
clust_eval = metrics_data["clustering_evaluation"]
sil_df = pd.DataFrame(
    {
        "k": [int(k) for k in clust_eval["silhouette_scores_by_k"].keys()],
        "silhouette_score": list(clust_eval["silhouette_scores_by_k"].values()),
        "inertia": list(clust_eval["inertias_by_k"].values()),
    }
)
display(sil_df)
display(pd.DataFrame(clust_eval["cluster_profiles"]).T)
display(Image(filename=str(PROJECT_ROOT / "reports/figures/clustering_silhouette_and_pca.png")))

### Analysis of Clustering Optimization & 2D PCA Projections
- **Optimal Cluster Count ($k^* = 6$)**: Across $k \in \{2..8\}$, the Silhouette Score rises steeply from **0.2774** ($k=2$) and **0.4787** ($k=4$) to **0.5562** ($k=5$) and peaks at **$k^* = 6$ with a Silhouette Score of 0.5663** before declining at $k=7$ (**0.5105**) and $k=8$ (**0.4442**).
- **Why $k=6$ Outperforms $k=5$**: Inspecting the cluster profiles reveals that $K$-Means naturally isolates the four applied enterprise domains into dedicated high-purity clusters—**Cluster 0 (`Healthcare`, 81.00% purity, 100 docs)**, **Cluster 1 (`Legal`, 78.85% purity, 104 docs)**, **Cluster 3 (`Technology`, 83.70% purity, 92 docs)**, and **Cluster 4 (`Finance`, 85.71% purity, 98 docs)**—while subdividing **`Science`** into two distinct physical sub-disciplines: **Cluster 2 (Earth, Ocean & Plasma Physics: `ocean`, `tokamak`, `atmospheric`, `vent`, 79.41% purity, 68 docs)** and **Cluster 5 (Quantum & Photovoltaic Physics: `solar`, `cell`, `perovskite`, `error`, `correction`, 81.58% purity, 38 docs)**.
- **2D PCA Projection Insights**: The first two principal components capture **50.54%** of total variance (`PCA 1 = 27.10%`, `PCA 2 = 23.44%`), clearly separating high-citation public research clusters (`Healthcare` and `Science`) from confidential regulatory clusters (`Finance` and `Legal`).

## 6. Keyword Search & TF-IDF Cosine Similarity Search Verification
Finally, we verify both retrieval engines (`keyword_search` via our Inverted Index and `similarity_search` via Cosine Similarity over $L_2$-normalized TF-IDF vectors) on a technical query.

In [ ]:
import joblib

bundle = joblib.load(PROJECT_ROOT / "models/document_intelligence.joblib")
search_engine = bundle["search_engine"]

query = "transformer neural network GPU inference optimization"
kw_df = pd.DataFrame(search_engine.keyword_search(query, top_k=4))[["rank", "doc_id", "title", "category", "cluster_id", "keyword_score", "matched_keywords"]]
sim_df = pd.DataFrame(search_engine.similarity_search(query, top_k=4))[["rank", "doc_id", "title", "category", "cluster_id", "similarity_percentage", "shared_tfidf_terms"]]

print("--- Top 4 Inverted-Index Keyword Search Results ---")
display(kw_df)
print("--- Top 4 TF-IDF Cosine Similarity Search Results ---")
display(sim_df)

### Analysis of Dual Search Retrieval
- **Inverted-Index Keyword Search**: Retrieves documents containing exact or lemmatized unigram/bigram query tokens (`gpu`, `inference`, `network`, `neural`, `neural network`, `transformer`), boosting phrase and title matches to surface `Technology` (`Cluster 3`) studies at the top.
- **Cosine Similarity Search**: Measures the angle between the $L_2$-normalized query TF-IDF vector and all 500 document vectors, ranking documents by lexical-semantic alignment even when only a subset of query terms is present.

### Q&A
- **How does the system preprocess raw documents and handle corrupted or incomplete data?**
  The preprocessing pipeline (`src/text_preprocessing.py`) audits the 510-row raw corpus, removing 5 schema-violating records (negative author experience `-8.5`, negative citations `-42.0`, empty text body, extreme experience `145.0` years, and invalid verification flag `9`) and 5 unlabeled records missing a `category` target. For the remaining 500 valid documents, it converts `published_date` strings into Unix timestamps, preserves the 37 rows (7.4%) with missing numerical metadata via median imputation and missingness indicators inside a leak-free `ColumnTransformer`, and normalizes text via lowercasing, stopword removal, and morphological lemmatization.
- **Which supervised classification model performs best for automated document categorization?**
  `Logistic Regression (L2)` is the optimal production model. While an unregularized Decision Tree severely overfits (`Train Macro F1 = 1.0000` vs. `Validation Macro F1 = 0.6014`), `Logistic Regression (L2)` achieves **0.9587 Validation ROC-AUC** and generalizes to the held-out test set with **85.33% Accuracy** (95% CI: `[77.33%, 92.00%]`), **0.8545 Macro F1** (95% CI: `[0.7724, 0.9203]`), and **0.9833 Macro ROC-AUC** (95% CI: `[0.9651, 0.9957]`) at a compact **55.21 KB** model size and **0.0675 ms** per-document inference latency.
- **How many natural document clusters exist in the corpus, and what do they represent?**
  Evaluating $K$-Means across $k \in \{2..8\}$ identifies an optimal **$k^* = 6$ clusters** with a peak **Silhouette Score of 0.5663** (compared to `0.5562` at $k=5$ and `0.2774` at $k=2$). Four clusters map directly to `Healthcare` (81.00% purity), `Legal` (78.85% purity), `Technology` (83.70% purity), and `Finance` (85.71% purity), while `Science` naturally bifurcates into Earth/Plasma Science (`Cluster 2`, 79.41% purity) and Quantum/Photovoltaic Physics (`Cluster 5`, 81.58% purity).

### Data Analysis Key Findings
- **Discriminative TF-IDF Signatures**: Fitting an 800-feature unigram/bigram `TfidfVectorizer` strictly on the 350-document training split isolates high-IDF domain vocabulary: `clinical` (`0.0697`) and `patient` (`0.0606`) in Healthcare; `regulatory` (`0.0604`) and `compliance` (`0.0548`) in Legal; `adequacy` (`0.0525`) and `central` (`0.0519`) in Finance; `superconduct` (`0.0595`) and `ocean` (`0.0557`) in Science; and `learn` (`0.0602`) and `zero` (`0.0535`) in Technology.
- **Subgroup Slice Performance**: Held-out test evaluation across metadata slices shows strong classification performance across security tiers and channels (`Confidential`: **87.50%** accuracy, `Regulatory_Filing`: **93.75%** accuracy), whereas interdisciplinary `Enterprise_Wiki` documents exhibit more cross-domain lexical overlap.
- **Dimensionality Reduction & Cluster Separation**: Projecting the normalized feature space into 2D via PCA captures **50.54%** of total variance in the first two components (`27.10%` and `23.44%`), visually confirming distinct cluster boundaries.

### Insights or Next Steps
- **Hybrid Retrieval Fusion**: Combine the Inverted-Index Keyword Score and TF-IDF Cosine Similarity Score using Reciprocal Rank Fusion (RRF) so exact regulatory/legal identifier lookups and broader conceptual queries are ranked in a single unified list.
- **Active Learning for Interdisciplinary Wiki Docs**: Route low-confidence predictions (`predicted probability < 60%`) on `Enterprise_Wiki` documents to human reviewers to expand training coverage on cross-departmental topics.